# Epic: Did that lemma come from speech or narrative?

Given a lemma, what is the likelihood that it came from direct speech versus narrative?

In [2]:
import numpy as np
import pandas as pd

narrative_df = pd.read_parquet("../parquet/homeric_narrative_dtm.parquet")
speeches_df = pd.read_parquet("../parquet/homeric_speeches_dtm.parquet")

In [65]:
n_lemmata_iliad_narrative = narrative_df["Iliad"].sum()
n_lemmata_odyssey_narrative = narrative_df["Odyssey"].sum()

print(f"Number of unique (non-stopword) lemmata in Iliadic narrative: {n_lemmata_iliad_narrative}")
print(f"Number of unique (non-stopword) lemmata in Odyssean narrative: {n_lemmata_odyssey_narrative}")
print("\n\n")

iliad_narrative_pcts = narrative_df["Iliad"] / n_lemmata_iliad_narrative
odyssey_narrative_pcts = narrative_df["Odyssey"] / n_lemmata_odyssey_narrative

print(iliad_narrative_pcts.sort_values(ascending=False).head(100))
iliad_narrative_pcts.sort_values(ascending=False).head(100).to_csv("../csv/iliad_narrative_top_100_lemmata.csv")
print("\n\n")
print(odyssey_narrative_pcts.sort_values(ascending=False).head(100))
odyssey_narrative_pcts.sort_values(ascending=False).head(100).to_csv("../csv/odyssey_narrative_top_100_lemmata.csv")

print("\n\n")

total_lemmata_narrative = n_lemmata_iliad_narrative + n_lemmata_odyssey_narrative
narrative_df["all"] = narrative_df["Iliad"] + narrative_df["Odyssey"]
all_pcts = narrative_df["all"] / total_lemmata_narrative

all_pcts.sort_values(ascending=False).head(100).to_csv("../csv/all_narrative_top_100_lemmata.csv")


Number of unique (non-stopword) lemmata in Iliadic narrative: 21455
Number of unique (non-stopword) lemmata in Odyssean narrative: 9646



lemma
ναῦς         0.008203
Ἕκτωρ        0.008110
υἱός         0.008063
Τρώς         0.007551
Ἀχαιός       0.007318
               ...   
Ἀντίλοχος    0.001585
ἀγαθός       0.001585
μακρός       0.001585
ὄρος         0.001585
αἷμα         0.001585
Name: Iliad, Length: 100, dtype: float64



lemma
Ὀδυσσεύς     0.019075
Τηλέμαχος    0.008501
χείρ         0.007983
βαίνω        0.007775
Ἀθήνη        0.007361
               ...   
αὐτίκος      0.001555
χέω          0.001555
κακός        0.001555
Μενέλαος     0.001555
δαίς         0.001555
Name: Odyssey, Length: 100, dtype: float64





In [66]:
n_lemmata_speeches = speeches_df.sum(axis=1).sum()
speeches_pcts = speeches_df.sum(axis=1) / n_lemmata_speeches

speeches_pcts.sort_values(ascending=False).head(100).to_csv("../csv/all_speeches_top_100_lemmata.csv")

In [67]:
n_lemmata_iliad_speeches = speeches_df["Iliad"].sum(axis=1).sum()
n_lemmata_odyssey_speeches = speeches_df["Odyssey"].sum(axis=1).sum()

iliad_speeches_pcts = speeches_df["Iliad"].sum(axis=1) / n_lemmata_iliad_speeches
odyssey_speeches_pcts = speeches_df["Odyssey"].sum(axis=1) / n_lemmata_odyssey_speeches

iliad_speeches_pcts.sort_values(ascending=False).head(100).to_csv("../csv/iliad_speeches_top_100_lemmata.csv")
odyssey_speeches_pcts.sort_values(ascending=False).head(100).to_csv("../csv/odyssey_speeches_top_100_lemmata.csv")


## Speech versus narrative 

In [20]:
df = pd.read_parquet("../parquet/homer_speech_narrative.parquet")

def build_totals(df: pd.DataFrame) -> dict[tuple[str, str], int]:
    return {key: int(df[key].sum()) for key in df.columns}

totals = build_totals(df)

In [21]:
def log_likelihood(a: pd.Series, b: pd.Series, total_a: int, total_b: int) -> pd.Series:
    """Dunning G² for each lemma. Positive = skewed toward corpus A."""
    N = total_a + total_b
    expected_a = (a + b) * total_a / N
    expected_b = (a + b) * total_b / N

    def safe_term(obs, exp):
        return np.where(obs > 0, obs * np.log(obs / exp), 0)

    g2 = 2 * (safe_term(a, expected_a) + safe_term(b, expected_b))
    return pd.Series(np.where(a / total_a >= b / total_b, g2, -g2), index=a.index)

In [22]:
totals_iliad = totals["iliad", "speech"] + totals["iliad", "narrative"]
totals_odyssey = totals["odyssey", "speech"] + totals["odyssey", "narrative"]

In [23]:
il = df["iliad"].sum(axis=1)
od = df["odyssey"].sum(axis=1)

loglikelihood_work = log_likelihood(il, od, totals_iliad, totals_odyssey)

/Users/pletcher/code/writing/articles/2026-06-13_ccc-2026/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pletcher/code/writing/articles/2026-06-13_ccc-2026/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [24]:
loglikelihood_work.sort_values(ascending=False).to_csv("../csv/iliad_vs_odyssey.csv")

In [25]:
speech = df.xs("speech", axis=1, level="register").sum(axis=1)
narrative = df.xs("narrative", axis=1, level="register").sum(axis=1)

totals_speech = totals["iliad", "speech"] + totals["odyssey", "speech"]
totals_narrative = totals["iliad", "narrative"] + totals["odyssey", "narrative"]

loglikelihood_register = log_likelihood(speech, narrative, totals_speech, totals_narrative)

/Users/pletcher/code/writing/articles/2026-06-13_ccc-2026/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [26]:
loglikelihood_register.sort_values(ascending=False).to_csv("../csv/speech_vs_narrative.csv")